# 07_assemble_submission_package

**Role.** Final package QA: tables, figures, supplementary files, metadata, and manuscript inserts.

**Pipeline version.** Reproducible scientific pipeline v2 for Dak Lak 2024 coffee mapping Paper 1.


In [ ]:
# =============================================================================
# REPRODUCIBILITY BOOTSTRAP: Coffee Paper 1 pipeline v2
# =============================================================================
from pathlib import Path
import os, sys, json, warnings
import numpy as np

# Locate project root robustly whether the notebook is opened from project root
# or from the notebooks/ folder.
_candidate_roots = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _candidate_roots if (p / "config" / "paper1_config.yaml").exists()), Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from coffeemap.config import load_config, ensure_project_dirs, class_info, class_colors, coffee_class_ids
from coffeemap.manifest import init_run_manifest, append_manifest_note
from coffeemap.plotting import set_publication_style

CONFIG = load_config(PROJECT_ROOT / "config" / "paper1_config.yaml")
PATHS = ensure_project_dirs(CONFIG, PROJECT_ROOT)
CLASS_INFO = class_info(CONFIG)
CLASS_COLORS = class_colors(CONFIG)
CLASS_IDS = sorted(CLASS_INFO.keys())
CLASS_NAMES = [CLASS_INFO[i] for i in CLASS_IDS]
COFFEE_CLASSES = coffee_class_ids(CONFIG)
RANDOM_SEED = int(CONFIG.get("project", {}).get("random_seed", 42))
np.random.seed(RANDOM_SEED)

TABLES_DIR = PATHS["tables_dir"]
FIGURES_DIR = PATHS["figures_dir"]
SUPPLEMENTARY_DIR = PATHS["supplementary_dir"]
METADATA_DIR = PATHS["metadata_dir"]
INPUT_DIR = PATHS["input_dir"]

NOTEBOOK_NAME = "06_assemble_submission_package.ipynb"
MANIFEST = init_run_manifest(CONFIG, PROJECT_ROOT, notebook_name=NOTEBOOK_NAME)
set_publication_style(font="Arial", dpi=600)

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Classes: {len(CLASS_IDS)} | Coffee classes: {COFFEE_CLASSES} | Random seed: {RANDOM_SEED}")


## Reproducibility contract

This notebook follows the project-level configuration in `config/paper1_config.yaml` and writes outputs only under `results/`.

Key safeguards used in this pipeline:

- class IDs, class names, colors, paths, random seed, and coffee class definitions come from one config file;
- each notebook refreshes `results/metadata/run_manifest.json`;
- feature selection must use training data only;
- validation data are reserved for final assessment;
- Olofsson-style estimates are reported as **area-weighted error-adjusted estimates** unless a mapped-class stratified area-assessment sample is available;
- RF uncertainty is interpreted as **RF vote-based class probability**, not calibrated posterior probability.


In [2]:
# =============================================================================
# Pipeline-level imports commonly used by downstream cells
# =============================================================================
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from coffeemap.io import find_file, read_table, write_table
from coffeemap.schema import (
    detect_column, detect_label_columns, assert_class_ids,
    class_count_table, warn_if_balanced, extract_probability_columns,
    assert_probability_matrix,
)
from coffeemap.metrics import classification_summary, overall_metrics, shannon_entropy, probability_margin
from coffeemap.validation import audit_validation_predictions
from coffeemap.olofsson import error_matrix_counts, area_adjustment, binary_coffee_area_adjustment

SEARCH_DIRS = [INPUT_DIR, PATHS["interim_dir"], TABLES_DIR, SUPPLEMENTARY_DIR, PROJECT_ROOT]
print("Reproducible pipeline helpers loaded.")


Reproducible pipeline helpers loaded.


In [ ]:
# =============================================================================
# EXPECTED CORE OUTPUTS
# =============================================================================
EXPECTED = {
    "Tables": [
        "Table1", "Table2", "Table3", "Table4", "Table5", "Table6",
    ],
    "Figures": [
        "Figure1", "Figure2", "Figure3", "Figure4", "Figure5", "Figure6",
        "Figure7", "Figure8", "Figure9", "Figure10",
    ],
    "Supplementary": [
        "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10",
        "Manifest", "Audit",
    ],
}

def folder_files(root):
    if not root.exists():
        return []
    return [p for p in root.rglob("*") if p.is_file()]

rows = []
for folder_name, keywords in EXPECTED.items():
    root = PROJECT_ROOT / folder_name
    files = folder_files(root)
    joined = "\n".join(str(p.name) for p in files)
    for kw in keywords:
        rows.append({
            "folder": folder_name,
            "expected_keyword": kw,
            "found": kw.lower() in joined.lower(),
            "matching_files": "; ".join([p.name for p in files if kw.lower() in p.name.lower()])[:500],
        })
qa = pd.DataFrame(rows)
qa.to_csv(SUPPLEMENTARY_DIR / "FinalPackage_QA_ExpectedOutputs.csv", index=False, encoding="utf-8-sig")
display(qa)
missing = qa.loc[~qa["found"]]
if len(missing):
    warnings.warn(f"{len(missing)} expected output keywords were not found. See FinalPackage_QA_ExpectedOutputs.csv")


In [ ]:
# =============================================================================
# OUTPUT MANIFEST
# =============================================================================
manifest_rows = []
for root in [TABLES_DIR, FIGURES_DIR, SUPPLEMENTARY_DIR]:
    if root.exists():
        for p in sorted(root.rglob("*")):
            if p.is_file():
                manifest_rows.append({
                    "folder": root.name,
                    "file": str(p.relative_to(root)),
                    "size_kb": round(p.stat().st_size / 1024, 1),
                })
manifest = pd.DataFrame(manifest_rows)
manifest_path = SUPPLEMENTARY_DIR / f"Manifest_{Path().resolve().name}.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(manifest.tail(30))
print("Manifest saved:", manifest_path)


In [5]:
# =============================================================================
# BUILD FINAL PACKAGE FOLDER
# =============================================================================
import shutil
from datetime import datetime

package_root = PROJECT_ROOT / "CoffeePaper1_FinalPackage"
if package_root.exists():
    shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)

for folder in [TABLES_DIR, FIGURES_DIR, SUPPLEMENTARY_DIR]:
    dst = package_root / folder.name
    if folder.exists():
        shutil.copytree(folder, dst, dirs_exist_ok=True)

readme = (
    "# Coffee Mapping Paper 1: final output package\n\n"
    f"Generated: {datetime.now().isoformat(timespec='seconds')}\n\n"
    "Folder contract:\n"
    "- Tables/: manuscript-ready tables.\n"
    "- Figures/: manuscript-ready main figures.\n"
    "- Supplementary/: supplementary tables, figures, audits, captions and manifests.\n\n"
    "Recommended manuscript order:\n"
    "1. Classification accuracy and confusion diagnostics.\n"
    "2. Feature importance and SHAP interpretation.\n"
    "3. Area-weighted error-adjusted estimates.\n"
    "4. District-level agreement with official statistics.\n"
    "5. Uncertainty and spatial robustness in Supplementary Material.\n\n"
    "Statistical note:\n"
    "Area-weighted error-adjusted estimates from Notebook 04 should not be called fully design-based "
    "unbiased estimates unless an independent map-stratified area-assessment sample is used.\n"
)
(package_root / "README_FinalPackage.md").write_text(readme, encoding="utf-8")
print("Final package folder created:", package_root)


Final package folder created: D:\2024_PhD_Research\Chap2_Mapping_Coffee\Code_workflow\CoffeePaper1_FinalPackage


In [6]:
# =============================================================================
# ZIP FINAL PACKAGE
# =============================================================================
import zipfile

zip_path = PROJECT_ROOT / "CoffeePaper1_FinalPackage.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in package_root.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(package_root.parent))

print("ZIP created:", zip_path)


ZIP created: D:\2024_PhD_Research\Chap2_Mapping_Coffee\Code_workflow\CoffeePaper1_FinalPackage.zip
